# Trouver les facteurs de la demande avec PROC GLMSELECT : pas à pas, LASSO et sélection ascendante validée


## Résumé exécutif

Une équipe d'analytique retail utilise **PROC GLMSELECT** pour découvrir quels leviers marketing et tarifaires font vraiment bouger les ventes hebdomadaires en unités, en séparant les véritables facteurs de la demande du bruit. La sélection pas à pas notée par le SBC, le LASSO avec validation croisée à 5 blocs, et une recherche ascendante validée sur une réserve retrouvent tous **le même ensemble de véritables facteurs** — prix propre, prix du concurrent, dépenses publicitaires, volume d'e-mails, promotions, jours fériés, la région Nord-Est et le canal en ligne — et chacun des quatre leurres plantés (`temp_f`, `noise1`, `noise2`, `noise3`) est automatiquement écarté.

Les méthodes s'accordent aussi étroitement sur les magnitudes : chacune estime un effet de prix propre proche de **-28 unités par dollar** et un effet de prix du concurrent proche de **+14**, les signes de substitution intégrés dans l'équation génératrice des données. Le seul désaccord se situe à la marge — le LASSO validé par validation croisée conserve en plus un petit contraste `Region=Midwest` statistiquement négligeable (estimation -8,3 avec une erreur type de 8,3) que la sélection pas à pas et la sélection ascendante écartent toutes les deux. Une liste de facteurs qui survit à trois philosophies de sélection différentes est bien plus fiable qu'une liste ajustée à une seule règle.


## Sources de données

Toutes les données de ce notebook sont **synthétiques** et générées en ligne (aucun fichier externe, graine `20260531`). Elles imitent une saison de panels de demande magasin-semaine pour un distributeur de biens de consommation.

| Jeu de données | Lignes | Grain | Variables clés |
|---------|------|-------|---------------|
| `demand` | 100 | Magasin-semaine | `units` (réponse : unités vendues par semaine) ; `price`, `competitor` (prix propre et prix du concurrent en rayon) ; `adspend`, `email` (pression média) ; `promo`, `holiday` (indicateurs d'événement) ; `region`, `channel` (effets CLASS) ; `temp_f`, `noise1`-`noise3` (prédicteurs leurres/non pertinents) |

`units` est construite à partir d'une équation linéaire connue afin que l'ensemble "correct" des facteurs soit vérifiable ; `temp_f` et les trois colonnes `noise` ne portent aucun signal réel et servent à tester si chaque méthode de sélection les écarte.


# Trouver les facteurs de la demande avec PROC GLMSELECT

Un category manager dispose de dizaines d'explicatives candidates pour les ventes hebdomadaires : prix propre, prix du concurrent, dépenses publicitaires, volume d'e-mails, promotions, jours fériés, région du magasin, canal de vente, et même la météo. Tout jeter dans une seule régression invite au surajustement et à des coefficients instables. **PROC GLMSELECT** automatise la recherche d'un modèle parcimonieux, en prenant en charge la sélection pas à pas, le LASSO, l'elastic net et la sélection par angle minimal, avec validation croisée et partition de réserve intégrées.

Dans ce notebook, nous allons :

1. Générer un panel magasin-semaine synthétique réaliste avec un ensemble *connu* de véritables facteurs (plus des variables leurres délibérées).
2. Exécuter une **sélection pas à pas** notée par le SBC.
3. Exécuter un **LASSO** avec validation croisée à 5 blocs.
4. Exécuter une **sélection ascendante validée sur une réserve de 30 %**.

Une bonne méthode de sélection devrait retrouver les véritables facteurs et écarter le bruit — voyons cela.


## 1. Générer un panel de demande synthétique

La réponse `units` est construite à partir d'une équation linéaire explicite, donc nous connaissons la vérité terrain : le prix et le prix du concurrent, les dépenses publicitaires, le volume d'e-mails, les indicateurs de promotion et de jour férié, ainsi que les effets de région et de canal comptent tous. Les variables `temp_f`, `noise1`, `noise2` et `noise3` sont de purs leurres sans relation réelle avec les ventes. Une graine `call streaminit` rend les données reproductibles.


In [1]:
/* ---------------------------------------------------------------
   Panel synthétique de demande magasin-semaine pour un distributeur
   de biens de consommation. 'units' suit une équation CONNUE ;
   temp_f et noise1-3 sont des leurres.
   --------------------------------------------------------------- */
DONNÉES demand;
    APPELER streaminit(20260531);
    LONGUEUR region $9 channel $8 promo $3;
    FAIRE store_week = 1 JUSQU_À 100;
        /* Répartition des régions */
        u1 = rand('uniform');
        SI u1 < 0.34 ALORS region = 'Northeast';
        SINON SI u1 < 0.67 ALORS region = 'Midwest';
        SINON region = 'South';

        /* Canal de vente */
        SI rand('uniform') < 0.45 ALORS channel = 'Store';
        SINON channel = 'Online';

        /* Prix et leviers média */
        price      = round(8  + rand('uniform') * 12, 0.01);
        competitor = round(8  + rand('uniform') * 12, 0.01);
        adspend    = round(rand('gamma', 2) * 500, 1);
        email      = round(rand('uniform') * 100, 1);

        /* Indicateurs d'événement et un leurre météo non pertinent */
        temp_f     = round(40 + rand('uniform') * 50, 0.1);
        holiday    = (rand('uniform') < 0.12);
        SI rand('uniform') < 0.40 ALORS promo = 'Yes';
        SINON promo = 'No';

        /* Prédicteurs de pur bruit (sans effet réel) */
        noise1 = rand('normal');
        noise2 = rand('normal');
        noise3 = rand('normal');

        /* Unités hebdomadaires vendues à partir d'une équation structurelle connue */
        units = 900
              - 28   * price
              + 14   * competitor
              + 0.06 * adspend
              + 1.2  * email
              + 55   * (promo = 'Yes')
              + 70   * holiday
              + 40   * (region = 'Northeast')
              + 25   * (channel = 'Online')
              + 30   * rand('normal');
        SI units < 0 ALORS units = 0;
        SORTIE;
    FIN;
    GARDER region channel promo price competitor adspend email temp_f
         holiday noise1 noise2 noise3 units;
EXÉCUTER;


NOTE: DATA demand


NOTE: Wrote demand (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds


## 2. Profiler les données

Avant de modéliser, vérifions que la réponse et les principales candidates continues sont sur des échelles raisonnables.


In [2]:
PROCÉDURE MOYENNES DONNÉES=demand n mean std min max maxdec=1;
    VAR units price competitor adspend email;
    ÉTIQUETTE units="Unités vendues (hebdomadaire)" price="Prix propre"
          competitor="Prix du concurrent" adspend="Dépenses publicitaires"
          email="Volume d'e-mails";
    TITRE "Demande hebdomadaire et facteurs candidats";
EXÉCUTER;

                                       Demande hebdomadaire et facteurs candidats                                       

                                                  The MEANS Procedure

 Variable    Label                                  N        Mean     Std Dev     Minimum     Maximum
 ----------------------------------------------------------------------------------------------------
 units       Unités vendues (hebdomadaire)        100       874.8       136.3       508.6      1162.3
 price       Prix propre                          100        14.0         3.4         8.0        19.9
 competitor  Prix du concurrent                   100        13.8         3.4         8.1        19.9
 adspend     Dépenses publicitaires               100       990.6       726.9        50.0      3358.0
 email       Volume d'e-mails                     100        45.5        26.4         0.0        99.0
 --------------------------------------------------------------------------------------------


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 3. Sélection pas à pas notée par le SBC

La sélection pas à pas ajoute et retire alternativement des effets, ici jugée par le **critère bayésien de Schwarz (SBC)** à la fois pour le test d'entrée (`select=sbc`) et pour le choix final du modèle (`choose=sbc`). Le SBC pénalise la complexité plus fortement que l'AIC, favorisant des modèles plus légers.

Instructions et options clés :

- `CLASS region channel promo / param=reference` déclare les prédicteurs catégoriels avec un codage à cellule de référence.
- `selection=stepwise(select=sbc choose=sbc)` pilote la recherche.
- `details=summary` imprime le résumé pas à pas de la sélection ; `stb` ajoute les coefficients standardisés pour que les effets d'échelles différentes soient comparables.
- `output out=demand_scored p=predicted r=residual` enregistre les valeurs ajustées et les résidus pour le scoring en aval.

Lisez le **résumé de sélection pas à pas** comme la trace de la recherche : elle part du modèle complet à 12 effets et *retire* des effets un par un, en écartant `noise1`, `noise2`, `temp_f`, le contraste `Region=Midwest`, puis `noise3`, car chaque retrait abaisse le SBC. Ce qui survit dans la table des **estimations des paramètres** est le modèle retenu.


In [3]:
PROCÉDURE glmselect DONNÉES=demand seed=20260531;
    CLASSE region channel promo / PARAM=reference;
    MODÈLE units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=stepwise(select=sbc choose=sbc)
          details=summary stb;
    SORTIE out=demand_scored p=predicted r=residual;
    ÉTIQUETTE units="Unités vendues (hebdomadaire)" region="Région" channel="Canal"
          promo="Promotion" price="Prix propre" competitor="Prix du concurrent"
          adspend="Dépenses publicitaires" email="Volume d'e-mails"
          temp_f="Température (F)" holiday="Jour férié"
          noise1="Bruit 1" noise2="Bruit 2" noise3="Bruit 3";
    TITRE "Sélection pas à pas des facteurs de la demande (SBC)";
EXÉCUTER;

                                       Demande hebdomadaire et facteurs candidats                                       

The GLMSELECT Procedure


Dependent Variable: UNITS Unités vendues (hebdomadaire)


Number of Observations Used: 100

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                 Stepwise Selection Summary                                                  

    Step    Action            Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)
--------  --------  ----------------  -----------------  --------  --------  --------  --------  --------  --------  --------
       1   Removed           Bruit 1                 12    0.9507    0.9439  707.0094  711.2420  713.1843  740.8766


NOTE: PROC GLMSELECT data=demand

NOTE: The data set WORK.DEMAND_SCORED has 100 observations and 15 variables.
NOTE: PROC GLMSELECT statement used.


## 4. LASSO avec validation croisée

Le LASSO ramène les coefficients vers zéro, réalisant simultanément sélection et régularisation. Plutôt que de s'arrêter à un critère fixe, nous laissons la **validation croisée à 5 blocs** choisir le point du chemin LASSO avec la meilleure erreur hors échantillon.

- `selection=lasso(choose=cv stop=none)` trace le chemin complet du LASSO et sélectionne l'étape optimale par VC.
- `cvmethod=random(5)` répartit les observations en 5 blocs aléatoires.

Le **résumé de sélection LASSO** montre l'ordre dans lequel les effets entrent à mesure que la pénalité se relâche : `price` en premier, puis `adspend`, `competitor`, la région Nord-Est, `email`, `promo` et `holiday` — les sept vrais signaux entrent avant tout leurre. La validation croisée choisit ensuite l'étape où le PRESS hors échantillon est le plus bas, et la table des **estimations des paramètres** qui en résulte conserve exactement les facteurs authentiques (plus un petit terme `Region=Midwest`) tout en excluant `temp_f`, `noise1`, `noise2` et `noise3`, qui n'entrent qu'à la toute fin du chemin.


In [4]:
PROCÉDURE glmselect DONNÉES=demand seed=20260531;
    CLASSE region channel promo / PARAM=reference;
    MODÈLE units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=lasso(choose=cv stop=none)
          cvmethod=random(5);
    ÉTIQUETTE units="Unités vendues (hebdomadaire)" region="Région" channel="Canal"
          promo="Promotion" price="Prix propre" competitor="Prix du concurrent"
          adspend="Dépenses publicitaires" email="Volume d'e-mails"
          temp_f="Température (F)" holiday="Jour férié"
          noise1="Bruit 1" noise2="Bruit 2" noise3="Bruit 3";
    TITRE "LASSO avec validation croisée à 5 blocs";
EXÉCUTER;

                                       Demande hebdomadaire et facteurs candidats                                       

The GLMSELECT Procedure


Dependent Variable: UNITS Unités vendues (hebdomadaire)


Number of Observations Used: 100

  Cross Validation Information   

                   Item     Value
-----------------------  --------
Cross Validation Method    Random
        Number of Folds         5

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                             LASSO Selection Summary                                                             

    Step    Action                   Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  ---------


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 5. Sélection ascendante validée sur une réserve

Une troisième stratégie, complémentaire : construire le modèle par **sélection ascendante** (les effets n'entrent jamais que, ils ne sortent jamais), mais s'arrêter là où la performance sur un échantillon de réserve indépendant est la meilleure — protégeant directement contre le surajustement.

- `partition fraction(validate=0.30)` réserve aléatoirement 30 % des lignes pour la validation, laissant 70 observations d'entraînement et 30 de validation.
- `selection=forward(select=aic choose=validate)` ajoute des effets par AIC mais choisit le modèle final selon l'erreur sur l'échantillon de validation.

La table des **fractions de partition** confirme la répartition 70/30. La sélection ascendante intègre ensuite des effets jusqu'à ce que l'erreur de validation cesse de s'améliorer ; les huit effets de la table finale des **estimations des paramètres** sont précisément les véritables facteurs, les quatre leurres n'étant jamais admis. Quand trois méthodes construites sur des principes différents aboutissent aux mêmes facteurs, le modèle est bien plus susceptible d'être réel qu'un artefact d'une seule règle de sélection.


In [5]:
PROCÉDURE glmselect DONNÉES=demand seed=20260531;
    CLASSE region channel promo / PARAM=reference;
    MODÈLE units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=forward(select=aic choose=validate);
    partition fraction(validate=0.30);
    ÉTIQUETTE units="Unités vendues (hebdomadaire)" region="Région" channel="Canal"
          promo="Promotion" price="Prix propre" competitor="Prix du concurrent"
          adspend="Dépenses publicitaires" email="Volume d'e-mails"
          temp_f="Température (F)" holiday="Jour férié"
          noise1="Bruit 1" noise2="Bruit 2" noise3="Bruit 3";
    TITRE "Sélection ascendante validée sur une réserve de 30 %";
EXÉCUTER;

                                       Demande hebdomadaire et facteurs candidats                                       

The GLMSELECT Procedure


Dependent Variable: UNITS Unités vendues (hebdomadaire)


Number of Observations Used: 70               
Number of Observations Used for Validation: 30

Partition Fractions 

      Role  Fraction
----------  --------
  Training    0.7000
Validation    0.3000
   Testing    0.0000

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                            Forward Selection Summary                                                            

    Step    Action                   Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 6. Interprétation des résultats

Les trois stratégies de sélection retrouvent le **même ensemble de véritables facteurs de la demande** et écartent chaque leurre. En lisant directement les tables des **estimations des paramètres** :

- **Le prix** est le facteur dominant. Son coefficient standardisé dans le modèle pas à pas est de **-0,70**, de loin le plus grand en magnitude, et la pente brute se situe entre **-27,8** (pas à pas et LASSO) et **-28,6** (ascendante) unités par dollar — presque exactement le -28 avec lequel les données ont été générées. **Le prix du concurrent** fait bouger la demande dans l'autre sens à environ **+14,4** dans les trois ajustements, le comportement de substitution qu'un category manager attend.
- **Les dépenses publicitaires** (environ +0,062 unité par dollar) et le **volume d'e-mails** (environ +1,2 unité par envoi) font tous deux augmenter les ventes, quantifiant la réponse aux médias.
- **Les promotions** et les **jours fériés** portent les plus grands effets d'événement : le contraste `Promo=No` se situe environ entre **-51 et -57** par rapport à une semaine promue, et le gain lié au jour férié est d'environ **+66 à +76** unités. La **région Nord-Est** (environ +49 à +55) et le **canal en ligne** (environ +24 à +32) portent des effets de base positifs.
- Fait crucial, chaque leurre — `temp_f`, `noise1`, `noise2`, `noise3` — est **écarté** par la sélection pas à pas et la sélection ascendante, et exclu du modèle LASSO choisi par VC. Chaque méthode a retrouvé le signal structurel et ignoré le bruit.

Les trois méthodes ne sont pas identiques à l'octet près : la sélection pas à pas (SBC) et la recherche ascendante validée sur réserve s'accordent sur les mêmes huit effets, tandis que le LASSO validé par validation croisée conserve en plus un petit contraste `Region=Midwest` (-8,3, erreur type 8,3) — une différence au niveau du bruit de fond plutôt qu'un désaccord de fond. Qu'une liste de facteurs survive au SBC pas à pas, au LASSO validé par validation croisée et à la sélection ascendante validée sur réserve est le véritable enseignement : un résultat robuste à trois philosophies de sélection différentes est bien plus crédible qu'un résultat ajusté à un seul critère. Avec `OUTPUT OUT=demand_scored` capturant les valeurs ajustées et les résidus, le même flux de travail s'étend naturellement au scoring du calendrier de prix et de promotions prévu pour le prochain trimestre.
